In [6]:
import numpy as np
import pathlib
from assumptions import demand, smr_tech, SMR_CF, solar_tech, wind_tech, battery
from model import GridSupply, KKSupply, VESupply, DatacenterModel

## 1. Load inputs
*Three 8760-row txt files from `1_VE.ipynb`. Solar and wind are normalised capacity factors (production / installed capacity). Prices are hourly DK weighted-average spot prices in €/MWh.*

In [7]:
pat = pathlib.Path("variation_patterns")

solar_cf = np.loadtxt(pat / "PV_VE_2025_2026.txt")
wind_cf  = np.loadtxt(pat / "WL_VE_2025_2026.txt")
prices   = np.loadtxt(pat / "wp_2025_2026.txt")

print(f"solar_cf : {solar_cf.shape}  mean={solar_cf.mean():.3f}  max={solar_cf.max():.3f}")
print(f"wind_cf  : {wind_cf.shape}   mean={wind_cf.mean():.3f}  max={wind_cf.max():.3f}")
print(f"prices   : {prices.shape}    mean={prices.mean():.1f}   min={prices.min():.1f}  max={prices.max():.1f}  €/MWh")

solar_cf : (8760,)  mean=512.821  max=3718.870
wind_cf  : (8760,)   mean=1119.573  max=3978.540
prices   : (8760,)    mean=609.0   min=-228.7  max=4352.7  €/MWh


## 2. Demand and grid
*`DatacenterDemand` sets the total load (1 GW flat) and the on-site fraction `x`. `GridSupply` wraps the price series and computes import costs given any on-site dispatch profile.*

In [8]:
grid = GridSupply(prices, demand)

print(f"Total demand   : {demand.demand_mw:,.0f} MW")
print(f"On-site floor  : {demand.floor_mw:,.0f} MW  (x = {demand.x:.0%})")
print(f"Grid cap       : {demand.grid_cap_mw:,.0f} MW")
print(f"Annual demand  : {demand.annual_mwh/1e6:.3f} TWh")

Total demand   : 1,000 MW
On-site floor  : 500 MW  (x = 50%)
Grid cap       : 500 MW
Annual demand  : 8.760 TWh


## 3. KK — nuclear on-site
*Constant output at `floor_mw`. Minimum feasible capacity by construction — no sizing needed. Capacity factor enters only the fuel-cost calculation (annual generation = capacity × CF × 8760).*

In [9]:
kk = KKSupply(smr_tech, demand, capacity_factor=SMR_CF)

print(f"Installed capacity : {kk.capacity_mw:,.0f} MW")
print(f"Annual generation  : {kk.capacity_mw * SMR_CF * demand.HOURS / 1e6:.3f} TWh")
print(f"Annual on-site cost: {kk.annual_cost() / 1e6:.1f} M€")

Installed capacity : 500 MW
Annual generation  : 3.942 TWh
Annual on-site cost: 294.1 M€


## 4. VE — solar + wind + battery on-site
*Bisects on per-technology capacity `C` (MW solar = MW wind) until on-site generation + battery dispatch meets `floor_mw` every hour. May take a few seconds.*

In [10]:
ve = VESupply(solar_cf, wind_cf, solar_tech, wind_tech, battery, demand)

print(f"Per-tech capacity  : {ve.capacity_mw:,.1f} MW  (solar + wind = {2*ve.capacity_mw:,.1f} MW total)")
print(f"Battery            : {demand.floor_mw:,.0f} MW / {demand.floor_mw * battery.storage_hours:,.0f} MWh")
print(f"Annual on-site cost: {ve.annual_cost() / 1e6:.1f} M€")

Per-tech capacity  : 77.5 MW  (solar + wind = 155.0 MW total)
Battery            : 500 MW / 2,000 MWh
Annual on-site cost: 1954.6 M€


## 5. Full model — KK vs VE
*Adds grid purchase costs to each on-site option and computes total annualised cost and system LCOE.*

In [11]:
model = DatacenterModel(demand, grid, kk, ve)
results = model.run()

for r in results.values():
    print(r)
    print(f"  on-site cost : {r.annual_onsite_cost/1e6:.1f} M€/yr")
    print(f"  grid cost    : {r.annual_grid_cost/1e6:.1f} M€/yr")
    print(f"  grid imports : {r.grid_import_mwh/1e6:.3f} TWh/yr")
    print()

KK: 500 MW on-site | LCOE 338.04 €/MWh | total 2961.3 M€/yr
  on-site cost : 294.1 M€/yr
  grid cost    : 2667.2 M€/yr
  grid imports : 4.380 TWh/yr

VE: 78 MW on-site | LCOE 223.87 €/MWh | total 1961.1 M€/yr
  on-site cost : 1954.6 M€/yr
  grid cost    : 6.6 M€/yr
  grid imports : 0.008 TWh/yr

